In [ ]:
import os
from sec_edgar_downloader import Downloader

In [ ]:
DOWNLOAD_DIR = "sec-edgar-filings"
if not os.path.exists(DOWNLOAD_DIR):
    os.makedirs(DOWNLOAD_DIR)

In [ ]:
email_address = "sude.tiras@sabanciuniv.edu"  # Replace with your email address
dl = Downloader(DOWNLOAD_DIR, email_address)

In [ ]:
"""
import os
import time
import requests
import pandas as pd
from tqdm import tqdm
from sec_edgar_downloader import Downloader

# Load tickers from CSV
file_path = "companies.csv"  # Update this path if needed
tickers_df = pd.read_csv(file_path, header=None)
tickers_df = tickers_df.head(500)
tickers = tickers_df.iloc[1:, 1].tolist()  # Extract tickers (ignoring header)

# Define forms and years to download
years = range(2023, 2025)  # Adjust as needed
forms = ["10-Q"]  # Forms to download

# Function to download filings for each company
def download_filings(dl, cik, ticker, forms, years):
    for form in forms:
        for year in years:
            print(f"Downloading {form} for {ticker} ({year})...")
            dl.get(form, cik, after=f"{year}-01-01", before=f"{year}-12-31")

# Download filings for each company
for ticker in tqdm(tickers, desc="Downloading SEC Filings"):
    try:
        download_filings(dl, ticker, ticker, forms, years)
        time.sleep(1)  # Avoid SEC rate limits
    except Exception as e:
        print(f"Error downloading {ticker}: {e}")

print("Download complete!")
"""

In [ ]:
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from sec_edgar_downloader import Downloader

# Load tickers from CSV
file_path = "companies.csv"  # Update this path if needed
tickers_df = pd.read_csv(file_path, header=None)
tickers_df = tickers_df.head(500)
tickers = tickers_df.iloc[1:, 1].tolist()  # Extract tickers (ignoring header)

# Define forms and years to download
years = range(2000, 2022)  # Adjust as needed
forms = ["10-Q"]  # Forms to download

# Function to download filings for each company
def download_filings(cik, ticker):
    email_address = "sude.tiras@sabanciuniv.edu"  # Replace with your email address
    dl = Downloader(DOWNLOAD_DIR, email_address)
    for form in forms:
        for year in years:
            try:
                print(f"Downloading {form} for {ticker} ({year})...")
                dl.get(form, cik, after=f"{year}-01-01", before=f"{year}-12-31")
            except Exception as e:
                print(f"Error downloading {form} for {ticker} in {year}: {e}")

# Multi-threaded download
def main():
    max_threads = 20  # Adjust for SEC rate limits
    with ThreadPoolExecutor(max_workers=max_threads) as executor:
        futures = {executor.submit(download_filings, ticker, ticker): ticker for ticker in tickers}

        for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading SEC Filings"):
            ticker = futures[future]
            try:
                future.result()  # Ensure exceptions are caught
            except Exception as e:
                print(f"Error downloading {ticker}: {e}")

    print("Download complete!")

if __name__ == "__main__":
    main()


In [ ]:
import os
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from sec_edgar_downloader import Downloader
import time

# Define the download directory
DOWNLOAD_DIR = "sec-edgar-filings"
FOLDER_1 = "C:/Users/sudet/Desktop/sec-edgar-filings"
FOLDER_2 = "C:/Users/sudet/Desktop/DONE-sec-edgar"

# Load tickers from CSV
file_path = "companies.csv"  # Update this path if needed
tickers_df = pd.read_csv(file_path, header=None)
tickers_df = tickers_df.head(500)
tickers = tickers_df.iloc[1:, 1].tolist()  # Extract tickers (ignoring header)

# Define forms and years to download
years = range(2000, 2022)  # Adjust as needed
forms = ["10-Q"]  # Forms to download

"""# Function to check if a filing already exists
def filing_exists(ticker, form, cik, year):
    paths_to_check = [
        os.path.join(FOLDER_1, ticker, form),
        os.path.join(FOLDER_2, ticker, form)
    ]
    
    for base_path in paths_to_check:
        if os.path.exists(base_path):
            # Check if any folder starts with CIK_YEAR
            for folder in os.listdir(base_path):
                if folder.startswith(f"{cik}-{str(year)[-2:]}"):  # Match "CIK_YY"
                    return True  # Found a matching folder
    
    return False  # Not found in either folder"""
    
def filing_exists(ticker, form, cik, year):
    """
    Check if the filing for a specific ticker, form, CIK, and year already exists 
    in either FOLDER_1 or FOLDER_2.
    """
    paths_to_check = [
        os.path.join(FOLDER_1, ticker, form),
        os.path.join(FOLDER_2, ticker, form)
    ]
    
    # Convert year to 2-digit format for folder name match (e.g., 2017 -> "17")
    year_suffix = str(year)[-2:]  

    for base_path in paths_to_check:
        if os.path.exists(base_path):
            print(f"Checking in: {base_path}")  # Debug print

            # Iterate over files/folders in the form directory
            for folder in os.listdir(base_path):
                print(f"Found folder: {folder}")  # Debug print
                
                # Expected format: CIK-YY-ACCESSION_NUM
                if folder.startswith(f"{cik}-{year_suffix}"):
                    print(f"Match found: {folder} for {ticker} in {base_path}")  # Debug print
                    return True  # Found a matching filing
            
    return False  # No matching filing found

# Function to download filings for each company
def download_filings(cik, ticker):
    email_address = "hitiras@hotmail.com"  # Replace with your email address
    dl = Downloader(DOWNLOAD_DIR, email_address)
    
    for form in forms:
        for year in years:
            if filing_exists(ticker, form, cik, year):
                print(f"Skipping {form} for {ticker} ({year}) - Already Downloaded")
                continue
                
            try:
                print(f"Downloading {form} for {ticker} ({year})...")
                dl.get(form, cik, after=f"{year}-01-01", before=f"{year}-12-31")
                time.sleep(1)  # Avoid SEC rate limits
            except Exception as e:
                print(f"Error downloading {form} for {ticker} in {year}: {e}")
                time.sleep(10)
                

# Multi-threaded download
def main():
    max_threads = 20  # Adjust for SEC rate limits
    with ThreadPoolExecutor(max_workers=max_threads) as executor:
        futures = {executor.submit(download_filings, ticker, ticker): ticker for ticker in tickers}

        for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading SEC Filings"):
            ticker = futures[future]
            try:
                future.result()  # Ensure exceptions are caught
            except Exception as e:
                print(f"Error downloading {ticker}: {e}")

    print("Download complete!")

if __name__ == "__main__":
    main()


In [ ]:
import os
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from sec_edgar_downloader import Downloader
import time

# Define download directories
DOWNLOAD_DIR = "sec-edgar-filings"
FOLDER_1 = "C:/Users/sudet/Desktop/sec-edgar-filings"
FOLDER_2 = "C:/Users/sudet/Desktop/DONE-sec-edgar"

# Load tickers from CSV
file_path = "companies.csv"  # Update this path if needed
companies_df = pd.read_csv(file_path, header=None)
companies_df = companies_df.iloc[500:750]
tickers = companies_df.iloc[1:, 1].tolist()  # Extract tickers (ignoring header)
cik_dict = {row[1]: str(row[0]).zfill(10) for _, row in companies_df.iloc[1:].iterrows()}  # Map Ticker -> Padded CIK

# Define forms and years to download
years = range(2000, 2025)  # Adjust as needed
forms = ["10-Q"]  # Forms to download
    
def filing_exists(ticker, form, cik, year):
    """
    Check if the filing for a specific ticker, form, CIK, and year already exists 
    in either FOLDER_1 or FOLDER_2.
    """
    paths_to_check = [
        os.path.join(FOLDER_1, ticker, form),
        os.path.join(FOLDER_2, ticker, form)
    ]
    
    # Ensure CIK is zero-padded to 10 digits
    cik_str = str(cik).zfill(10)  # Ensures leading zeros (e.g., 320193 -> 0000320193)
    year_suffix = str(year)[-2:]  # Convert year to 2-digit format (e.g., 2017 -> "17")

    print(f"\nChecking for {ticker} ({form}, {year}) - CIK: {cik_str}")  # Debug Print

    for base_path in paths_to_check:
        print(f"📂 Looking in: {base_path}")  # Debug print

        if not os.path.exists(base_path):
            print(f"Path does not exist: {base_path}")
            continue  # Skip to the next folder if this one doesn't exist

        found_folders = os.listdir(base_path)
        print(f"📌 Found {len(found_folders)} folders in {base_path}: {found_folders}")  # Debug print

        for folder in found_folders:
            #print(f"   📁 Checking folder: {folder}")  # Debug print
            
            # Expected format: "0000320193-17-000009"
            if folder.startswith(f"{cik_str}-{year_suffix}"):
                print(f"✅ Match found: {folder} for {ticker} in {base_path}")
                return True  # Found a matching filing

    print(f"No match found for {ticker} in {paths_to_check}")  # Debugging output
    return False  # No matching filing found
                
def download_filings(cik, ticker):
    email_address = "sude.tiras@sabanciuniv.edu"  # Replace with your email address
    dl = Downloader(DOWNLOAD_DIR, email_address)

    for form in forms:
        for year in years:
            if filing_exists(ticker, form, cik, year):
                print(f"Skipping {form} for {ticker} ({year}) - Already Downloaded")
                continue

            try:
                print(f"Downloading {form} for {ticker} ({year})...")
                time.sleep(1)  # Avoid SEC rate limits
                dl.get(form, cik, after=f"{year}-01-01", before=f"{year}-12-31")
                #time.sleep(5)  # Avoid SEC rate limits
                
            except Exception as e:
                print(f"Error downloading {form} for {ticker} in {year}: {e}")
                time.sleep(30)  # Sleep longer if an error occurs
 

# Multi-threaded download
def main():
    max_threads = 20  # Reduce concurrency to prevent SEC rate limits
    with ThreadPoolExecutor(max_workers=max_threads) as executor:
        futures = {executor.submit(download_filings, cik_dict[ticker], ticker): ticker for ticker in tickers}

        for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading SEC Filings"):
            ticker = futures[future]
            try:
                future.result()  # Ensure exceptions are caught
            except Exception as e:
                print(f"Error downloading {ticker}: {e}")
                time.sleep(30)

    print("Download complete!")


if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd 
import os
# Paths
DOWNLOAD_DIR = "C:/Users/sudet/Desktop/finai/finai/Data Collection/sec-edgar-filings"
file_path = "companies.csv"  # Update with the actual path to your CSV file

# Load CIK-to-Ticker mapping from CSV
companies_df = pd.read_csv(file_path, header=None)
cik_dict = {str(row[0]).zfill(10): row[1] for _, row in companies_df.iloc[1:].iterrows()}  # CIK -> Ticker

# Rename CIK folders to Ticker names
for cik_folder in os.listdir(DOWNLOAD_DIR):
    cik_path = os.path.join(DOWNLOAD_DIR, cik_folder)

    # Ensure it's a directory and a valid CIK
    if os.path.isdir(cik_path) and cik_folder in cik_dict:
        ticker_name = cik_dict[cik_folder]  # Get corresponding ticker
        new_path = os.path.join(DOWNLOAD_DIR, ticker_name)

        if not os.path.exists(new_path):  # Avoid overwriting existing folders
            os.rename(cik_path, new_path)
            print(f"✅ Renamed {cik_folder} ➝ {ticker_name}")
        else:
            print(f"⚠️ Skipped {cik_folder} (Ticker {ticker_name} already exists!)")
    else:
        print(f"❌ Skipped {cik_folder} (No matching ticker found)")
